In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import URLError, HTTPError
from time import perf_counter

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
YEAR = 2024
OUTPUT_DIR = Path("data")
MONTHS = range(1, 13)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

success = 0
failed = 0

for month in MONTHS:
    month_str = f"{month:02d}"
    filename = f"yellow_tripdata_{YEAR}-{month_str}.parquet"
    url = f"{BASE_URL}/{filename}"
    output_path = OUTPUT_DIR / filename

    if output_path.exists():
        print(f"✓ Skipping {filename} (already exists)")
        success += 1
        continue

    print(f"Downloading {filename}...")

    try:
        start = perf_counter()
        urlretrieve(url, output_path)

        elapsed = perf_counter() - start
        size_mb = output_path.stat().st_size / (1024 * 1024)

        print(f"✓ Done ({size_mb:.1f} MB in {elapsed:.1f}s)")
        success += 1

    except HTTPError as e:
        print(f"✗ HTTP error: {e.code}")
        failed += 1

    except URLError as e:
        print(f"✗ Network error: {e.reason}")
        failed += 1

    except Exception as e:
        print(f"✗ Unexpected error: {e}")
        failed += 1

        if output_path.exists():
            output_path.unlink()

print(f"\nFinished — Success: {success}, Failed: {failed}")

✓ Skipping yellow_tripdata_2024-01.parquet (already exists)
✓ Skipping yellow_tripdata_2024-02.parquet (already exists)
✓ Skipping yellow_tripdata_2024-03.parquet (already exists)
✓ Skipping yellow_tripdata_2024-04.parquet (already exists)
✓ Skipping yellow_tripdata_2024-05.parquet (already exists)
✓ Skipping yellow_tripdata_2024-06.parquet (already exists)
✓ Skipping yellow_tripdata_2024-07.parquet (already exists)
✓ Skipping yellow_tripdata_2024-08.parquet (already exists)
✓ Skipping yellow_tripdata_2024-09.parquet (already exists)
✓ Skipping yellow_tripdata_2024-10.parquet (already exists)
✓ Skipping yellow_tripdata_2024-11.parquet (already exists)
✓ Skipping yellow_tripdata_2024-12.parquet (already exists)

Finished — Success: 12, Failed: 0


## Part 1: Spark RDDs

This section uses January 2024 taxi trips to practice Spark RDD operations before moving to DataFrames.


### 1.1 Load January as an RDD


In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Assignment2")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 12:50:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
jan_path = "data/yellow_tripdata_2024-01.parquet"

jan_df = spark.read.parquet(jan_path)
jan_rdd = jan_df.rdd

first_three = jan_rdd.take(3)
for i, row in enumerate(first_three, start=1):
    print(f"Element {i} type: {type(row)}")
    print(row)
    print()


[Stage 2:============================================>              (3 + 1) / 4]

Element 1 type: <class 'pyspark.sql.types.Row'>
Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0)

Element 2 type: <class 'pyspark.sql.types.Row'>
Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 3), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 0, 9, 36), passenger_count=1, trip_distance=1.8, RatecodeID=1, store_and_fwd_flag='N', PULocationID=140, DOLocationID=236, payment_type=1, fare_amount=10.0, extra=3.5, mta_tax=0.5, tip_amount=3.75, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=18.75, congestion_surcharge=2.5, Airport_fee=0.0)

Element 3 type: <class 'pyspark.sq

Each RDD element is a Spark `Row` object. It behaves like a structured record, but the RDD API does not automatically understand or optimize the column operations the way the DataFrame API does.


### 1.2 RDD Transformations


In [9]:
total_trips = jan_rdd.count()
print(f"Total trips in January 2024: {total_trips:,}")


[Stage 3:===============================================>         (10 + 2) / 12]

Total trips in January 2024: 2,964,624


In [10]:
high_passenger_trips = jan_rdd.filter(
    lambda row: row.passenger_count is not None and row.passenger_count > 4
)

high_passenger_count = high_passenger_trips.count()
print(f"Trips with more than 4 passengers: {high_passenger_count:,}")


[Stage 4:===============================================>         (10 + 2) / 12]

Trips with more than 4 passengers: 55,919


In [11]:
distance_totals = (
    jan_rdd
    .filter(lambda row: row.trip_distance is not None)
    .map(lambda row: (float(row.trip_distance), 1))
    .reduce(lambda left, right: (left[0] + right[0], left[1] + right[1]))
)

average_trip_distance = distance_totals[0] / distance_totals[1]
print(f"Average trip distance: {average_trip_distance:.2f} miles")


[Stage 5:===============================================>         (10 + 2) / 12]

Average trip distance: 3.65 miles


### 1.3 WordCount-style Aggregation


In [ ]:
revenue_by_pickup_location = (
    jan_rdd
    .filter(lambda row: row.PULocationID is not None and row.total_amount is not None)
    .map(lambda row: (int(row.PULocationID), float(row.total_amount)))
    .reduceByKey(lambda left, right: left + right)
)

top_5_pickup_locations = revenue_by_pickup_location.takeOrdered(
    5,
    key=lambda item: -item[1],
)

print("Top 5 pickup locations by total revenue:")
for rank, (location_id, revenue) in enumerate(top_5_pickup_locations, start=1):
    print(f"{rank}. PULocationID {location_id}: ${revenue:,.2f}")


### 1.4 Reflection

RDDs make the individual transformation steps explicit, but they require more manual work: we have to handle nulls ourselves, convert values, build key-value pairs, and define reductions. After Part 2, we will compare this with the equivalent DataFrame code, which should be shorter, more readable for structured data, and easier for Spark to optimize automatically.


## Part 2: DataFrames - Cleaning and Transforming

This section loads the full year of 2024 yellow taxi trips, explores data quality, cleans invalid records, creates derived features, and writes the cleaned dataset for the SQL and performance parts.


### 2.1 Load and Combine


In [14]:
from pyspark.sql import functions as F
from glob import glob

files = glob("data/yellow_tripdata_2024-*.parquet")
taxi_df = spark.read.parquet(*files)

taxi_df.printSchema()
total_rows = taxi_df.count()
total_columns = len(taxi_df.columns)

print(f"Total rows: {total_rows:,}")
print(f"Total columns: {total_columns:,}")


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

Total rows: 41,169,720
Total columns: 19


### 2.2 Data Exploration


In [15]:
print(f"Rows: {total_rows:,}")
print(f"Columns: {total_columns:,}")

print("\nColumn names:")
for column_name in taxi_df.columns:
    print(f"- {column_name}")


Rows: 41,169,720
Columns: 19

Column names:
- VendorID
- tpep_pickup_datetime
- tpep_dropoff_datetime
- passenger_count
- trip_distance
- RatecodeID
- store_and_fwd_flag
- PULocationID
- DOLocationID
- payment_type
- fare_amount
- extra
- mta_tax
- tip_amount
- tolls_amount
- improvement_surcharge
- total_amount
- congestion_surcharge
- Airport_fee


In [16]:
null_counts = (
    taxi_df
    .select([
        F.count(F.when(F.col(column_name).isNull(), column_name)).alias(column_name)
        for column_name in taxi_df.columns
    ])
    .first()
    .asDict()
)

columns_with_nulls = {
    column_name: count
    for column_name, count in null_counts.items()
    if count > 0
}

if columns_with_nulls:
    print("Columns with null values:")
    for column_name, count in sorted(columns_with_nulls.items(), key=lambda item: item[1], reverse=True):
        print(f"{column_name}: {count:,}")
else:
    print("No null values found.")


[Stage 18:==========================>                              (8 + 9) / 17]

Columns with null values:
passenger_count: 4,091,232
RatecodeID: 4,091,232
store_and_fwd_flag: 4,091,232
congestion_surcharge: 4,091,232
Airport_fee: 4,091,232


In [21]:
date_range = taxi_df.select(
    F.min("tpep_pickup_datetime").alias("min_pickup_datetime"),
    F.max("tpep_pickup_datetime").alias("max_pickup_datetime"),
).first()

pickup_outliers = taxi_df.filter(
    (F.col("tpep_pickup_datetime") < F.lit("2024-01-01"))
    | (F.col("tpep_pickup_datetime") >= F.lit("2025-01-01"))
).count()

print(f"Pickup datetime range: {date_range.min_pickup_datetime} to {date_range.max_pickup_datetime}")
print(f"Rows with pickup dates outside 2024: {pickup_outliers:,}")


[Stage 38:================>                                       (5 + 12) / 17]

Pickup datetime range: 2002-12-31 16:46:07 to 2026-06-26 23:53:12
Rows with pickup dates outside 2024: 56


In [23]:
payment_type_counts = (
    taxi_df
    .groupBy("payment_type")
    .count()
    .orderBy("payment_type")
)

payment_type_counts.show(truncate=False)

# 0 = Flex Fare trip
# 1 = Credit card
# 2 = Cash
# 3 = No charge
# 4 = Dispute
# 5 = Unknown
# 6 = Voided trip

+------------+--------+
|payment_type|count   |
+------------+--------+
|0           |4091232 |
|1           |30452159|
|2           |5540088 |
|3           |291743  |
|4           |794494  |
|5           |4       |
+------------+--------+



### 2.3 Data Cleaning


In [24]:
cleaning_steps = []

def apply_cleaning_step(input_df, description, condition):
    before_count = input_df.count()
    output_df = input_df.filter(condition)
    after_count = output_df.count()
    removed_count = before_count - after_count
    removed_percentage = (removed_count / before_count * 100) if before_count else 0
    cleaning_steps.append((description, before_count, after_count, removed_count, removed_percentage))
    print(f"{description}: removed {removed_count:,} rows ({removed_percentage:.4f}%)")
    return output_df

df_clean = taxi_df
df_clean = apply_cleaning_step(
    df_clean,
    "Remove trip_distance <= 0",
    F.col("trip_distance") > 0,
)
df_clean = apply_cleaning_step(
    df_clean,
    "Remove total_amount <= 0",
    F.col("total_amount") > 0,
)
df_clean = apply_cleaning_step(
    df_clean,
    "Remove null or zero passenger_count",
    F.col("passenger_count").isNotNull() & (F.col("passenger_count") > 0),
)
df_clean = apply_cleaning_step(
    df_clean,
    "Remove trip_distance > 200",
    F.col("trip_distance") <= 200,
)
df_clean = apply_cleaning_step(
    df_clean,
    "Remove total_amount > 5000",
    F.col("total_amount") <= 5000,
)
df_clean = apply_cleaning_step(
    df_clean,
    "Remove pickup dates outside 2024",
    (F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    & (F.col("tpep_pickup_datetime") < F.lit("2025-01-01")),
)

cleaned_rows = df_clean.count()
total_removed = total_rows - cleaned_rows
total_removed_percentage = total_removed / total_rows * 100

print("\nCleaning summary:")
print(f"Original rows: {total_rows:,}")
print(f"Cleaned rows: {cleaned_rows:,}")
print(f"Total removed: {total_removed:,} ({total_removed_percentage:.4f}%)")


Remove trip_distance <= 0: removed 776,305 rows (1.8856%)


Remove total_amount <= 0: removed 560,582 rows (1.3878%)


Remove null or zero passenger_count: removed 4,200,008 rows (10.5441%)


Remove trip_distance > 200: removed 266 rows (0.0007%)


Remove total_amount > 5000: removed 2 rows (0.0000%)


Remove pickup dates outside 2024: removed 55 rows (0.0002%)


[Stage 83:================>                                       (5 + 12) / 17]


Cleaning summary:
Original rows: 41,169,720
Cleaned rows: 35,632,502
Total removed: 5,537,218 (13.4497%)


In [28]:
cleaning_summary_df.orderBy("step").select(
    "step",
    F.format_number("rows_before", 0).alias("rows_before"),
    F.format_number("rows_after", 0).alias("rows_after"),
    F.format_number("rows_removed", 0).alias("rows_removed"),
    F.concat(F.format_number("percent_removed", 4), F.lit("%")).alias("percent_removed")
).show(truncate=False)


+-----------------------------------+-----------+----------+------------+---------------+
|step                               |rows_before|rows_after|rows_removed|percent_removed|
+-----------------------------------+-----------+----------+------------+---------------+
|Remove null or zero passenger_count|39,832,833 |35,632,825|4,200,008   |10.5441%       |
|Remove pickup dates outside 2024   |35,632,557 |35,632,502|55          |0.0002%        |
|Remove total_amount <= 0           |40,393,415 |39,832,833|560,582     |1.3878%        |
|Remove total_amount > 5000         |35,632,559 |35,632,557|2           |0.0000%        |
|Remove trip_distance <= 0          |41,169,720 |40,393,415|776,305     |1.8856%        |
|Remove trip_distance > 200         |35,632,825 |35,632,559|266         |0.0007%        |
+-----------------------------------+-----------+----------+------------+---------------+



### 2.4 Feature Engineering


In [30]:
df_enriched = (
    df_clean
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60,
    )
    .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", ((F.dayofweek("tpep_pickup_datetime") + 5) % 7) + 1)
    .withColumn("month", F.month("tpep_pickup_datetime"))
    .withColumn(
        "tip_percentage",
        F.when(F.col("fare_amount") > 0, F.col("tip_amount") / F.col("fare_amount") * 100).otherwise(0.0),
    )
    .withColumn(
        "speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance") / (F.col("trip_duration_minutes") / 60),
        ),
    )
)

before_speed_filter = df_enriched.count()
df_enriched = df_enriched.filter(F.col("speed_mph").isNull() | (F.col("speed_mph") <= 100))
after_speed_filter = df_enriched.count()
speed_rows_removed = before_speed_filter - after_speed_filter

print(f"Rows removed by speed_mph > 100 filter: {speed_rows_removed:,}")
print(f"Rows remaining after feature engineering filters: {after_speed_filter:,}")

df_enriched.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    F.round("trip_distance", 4).alias("trip_distance"),
    F.round("trip_duration_minutes", 4).alias("trip_duration_minutes"),
    "hour_of_day",
    "day_of_week",
    "month",
    F.round("tip_percentage", 4).alias("tip_percentage"),
    F.round("speed_mph", 4).alias("speed_mph")
).show(10, truncate=False)


Rows removed by speed_mph > 100 filter: 8,938
Rows remaining after feature engineering filters: 35,623,564
+--------------------+---------------------+-------------+---------------------+-----------+-----------+-----+--------------+---------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_minutes|hour_of_day|day_of_week|month|tip_percentage|speed_mph|
+--------------------+---------------------+-------------+---------------------+-----------+-----------+-----+--------------+---------+
|2024-01-01 00:57:55 |2024-01-01 01:17:43  |1.72         |19.8                 |0          |1          |1    |0.0           |5.2121   |
|2024-01-01 00:03:00 |2024-01-01 00:09:36  |1.8          |6.6                  |0          |1          |1    |37.5          |16.3636  |
|2024-01-01 00:17:06 |2024-01-01 00:35:01  |4.7          |17.9167              |0          |1          |1    |12.8755       |15.7395  |
|2024-01-01 00:36:38 |2024-01-01 00:44:56  |1.4          |8.3                

### 2.5 Aggregations


In [32]:
hourly_trip_summary = (
    df_enriched
    .groupBy("hour_of_day")
    .agg(
        F.round(F.avg("trip_distance"), 4).alias("avg_trip_distance"),
        F.round(F.avg("fare_amount"), 4).alias("avg_fare_amount"),
        F.count("*").alias("trip_count"),
    )
    .orderBy("hour_of_day")
)

hourly_trip_summary.show(24, truncate=False)

print("Hours with longest average trips:")
hourly_trip_summary.orderBy(F.desc("avg_trip_distance")).show(5, truncate=False)

print("Hours with most expensive average fares:")
hourly_trip_summary.orderBy(F.desc("avg_fare_amount")).show(5, truncate=False)


+-----------+-----------------+---------------+----------+
|hour_of_day|avg_trip_distance|avg_fare_amount|trip_count|
+-----------+-----------------+---------------+----------+
|0          |3.994            |20.3094        |954716    |
|1          |3.4516           |18.0657        |624162    |
|2          |3.0933           |16.5795        |405911    |
|3          |3.3698           |17.5814        |260011    |
|4          |4.9642           |23.8163        |166813    |
|5          |6.3389           |28.515         |196094    |
|6          |4.9429           |23.1423        |458702    |
|7          |3.7435           |19.536         |927136    |
|8          |3.2246           |18.5762        |1301596   |
|9          |3.1321           |18.65          |1508323   |
|10         |3.1565           |19.0391        |1663985   |
|11         |3.1306           |19.3041        |1806892   |
|12         |3.2214           |19.7308        |1960838   |
|13         |3.4126           |20.4608        |2033246  

+-----------+-----------------+---------------+----------+
|hour_of_day|avg_trip_distance|avg_fare_amount|trip_count|
+-----------+-----------------+---------------+----------+
|5          |6.3389           |28.515         |196094    |
|4          |4.9642           |23.8163        |166813    |
|6          |4.9429           |23.1423        |458702    |
|23         |4.0789           |20.9633        |1398690   |
|0          |3.994            |20.3094        |954716    |
+-----------+-----------------+---------------+----------+
only showing top 5 rows
Hours with most expensive average fares:


[Stage 126:================================>                      (10 + 7) / 17]

+-----------+-----------------+---------------+----------+
|hour_of_day|avg_trip_distance|avg_fare_amount|trip_count|
+-----------+-----------------+---------------+----------+
|5          |6.3389           |28.515         |196094    |
|4          |4.9642           |23.8163        |166813    |
|6          |4.9429           |23.1423        |458702    |
|16         |3.5587           |21.2992        |2283904   |
|14         |3.5698           |21.2518        |2183202   |
+-----------+-----------------+---------------+----------+
only showing top 5 rows


In [38]:
monthly_revenue = (
    df_enriched
    .groupBy("month")
    .agg(F.sum("total_amount").alias("total_revenue"))
    .orderBy("month")
)

monthly_revenue.select(
    "month",
    F.format_number("total_revenue", 4).alias("total_revenue")
).show(12, truncate=False)


[Stage 144:================>                                      (5 + 12) / 17]

+-----+---------------+
|month|total_revenue  |
+-----+---------------+
|1    |74,618,210.9899|
|2    |74,443,845.3399|
|3    |86,015,272.0099|
|4    |85,798,744.9199|
|5    |93,709,999.4599|
|6    |87,459,995.0499|
|7    |78,999,762.3899|
|8    |77,221,241.4699|
|9    |91,313,582.9899|
|10   |98,708,759.0099|
|11   |90,777,691.5599|
|12   |94,453,498.6799|
+-----+---------------+



In [41]:
daily_trip_counts = (
    df_enriched
    .groupBy("day_of_week")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("day_of_week")
)

daily_trip_counts.show(7, truncate=False)

print("Busiest day of week:")
daily_trip_counts.orderBy(F.desc("trip_count")).show(1, truncate=False)


+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|1          |4547041   |
|2          |5200650   |
|3          |5369091   |
|4          |5584259   |
|5          |5282992   |
|6          |5236548   |
|7          |4402983   |
+-----------+----------+

Busiest day of week:


[Stage 162:================================>                      (10 + 7) / 17]

+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|4          |5584259   |
+-----------+----------+
only showing top 1 row


In [43]:
tip_by_payment_type = (
    df_enriched
    .groupBy("payment_type")
    .agg(
        F.round(F.avg("tip_percentage"), 4).alias("avg_tip_percentage"),
        F.count("*").alias("trip_count"),
    )
    .orderBy(F.desc("avg_tip_percentage"))
)

tip_by_payment_type.show(truncate=False)


[Stage 168:================>                                      (5 + 12) / 17]

+------------+------------------+----------+
|payment_type|avg_tip_percentage|trip_count|
+------------+------------------+----------+
|1           |25.0977           |29870931  |
|4           |0.063             |383997    |
|3           |0.0614            |148514    |
|2           |0.0015            |5220122   |
+------------+------------------+----------+



### 2.6 Write Cleaned Data


In [44]:
clean_output_path = "data/nyc_taxi_2024_clean.parquet"

(
    df_enriched
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(clean_output_path)
)

print(f"Wrote cleaned and enriched data to {clean_output_path}")


Wrote cleaned and enriched data to data/nyc_taxi_2024_clean.parquet


## Part 3: Spark SQL and Window Functions

This section registers the cleaned taxi DataFrame as a temporary SQL view and answers the assignment questions using Spark SQL.


In [45]:
# If the kernel was restarted, reload the cleaned data written in Part 2.
try:
    df_enriched
except NameError:
    df_enriched = spark.read.parquet("data/nyc_taxi_2024_clean.parquet")

df_clean = df_enriched
df_clean.createOrReplaceTempView("trips")

print(f"Registered trips view with {df_clean.count():,} rows and {len(df_clean.columns):,} columns.")


[Stage 172:================================>                      (10 + 7) / 17]

Registered trips view with 35,623,564 rows and 25 columns.


### 3.1 SQL Basics


In [46]:
top_10_busiest_pickups_sql = spark.sql("""
    SELECT
        PULocationID,
        COUNT(*) AS trip_count
    FROM trips
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
""")

top_10_busiest_pickups_sql.show(truncate=False)


[Stage 175:======================>                                (7 + 10) / 17]

+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|132         |1856382   |
|237         |1766762   |
|161         |1737647   |
|236         |1557671   |
|162         |1309910   |
|186         |1269313   |
|230         |1250688   |
|138         |1245356   |
|142         |1182683   |
|163         |1063249   |
+------------+----------+



In [47]:
payment_type_amounts_sql = spark.sql("""
    SELECT
        payment_type,
        ROUND(AVG(fare_amount), 4) AS avg_fare_amount,
        ROUND(AVG(tip_amount), 4) AS avg_tip_amount,
        ROUND(AVG(total_amount), 4) AS avg_total_amount,
        COUNT(*) AS trip_count
    FROM trips
    GROUP BY payment_type
    ORDER BY payment_type
""")

payment_type_amounts_sql.show(truncate=False)


[Stage 178:================>                                      (5 + 12) / 17]

+------------+---------------+--------------+----------------+----------+
|payment_type|avg_fare_amount|avg_tip_amount|avg_total_amount|trip_count|
+------------+---------------+--------------+----------------+----------+
|1           |19.7467        |4.358         |29.7534         |29870931  |
|2           |19.6366        |2.0E-4        |24.9589         |5220122   |
|3           |19.9421        |0.0064        |25.2672         |148514    |
|4           |22.2561        |0.0138        |27.9135         |383997    |
+------------+---------------+--------------+----------------+----------+



In [48]:
long_distance_pickups_sql = spark.sql("""
    WITH overall AS (
        SELECT AVG(trip_distance) AS overall_avg_trip_distance
        FROM trips
    ),
    pickup_averages AS (
        SELECT
            PULocationID,
            AVG(trip_distance) AS avg_trip_distance,
            COUNT(*) AS trip_count
        FROM trips
        GROUP BY PULocationID
    )
    SELECT
        p.PULocationID,
        ROUND(p.avg_trip_distance, 4) AS avg_trip_distance,
        ROUND(o.overall_avg_trip_distance, 4) AS overall_avg_trip_distance,
        p.trip_count
    FROM pickup_averages p
    CROSS JOIN overall o
    WHERE p.avg_trip_distance > 2 * o.overall_avg_trip_distance
    ORDER BY p.avg_trip_distance DESC
""")

long_distance_pickups_sql.show(20, truncate=False)


[Stage 182:===================================>                   (11 + 6) / 17]

+------------+-----------------+-------------------------+----------+
|PULocationID|avg_trip_distance|overall_avg_trip_distance|trip_count|
+------------+-----------------+-------------------------+----------+
|44          |22.875           |3.4374                   |8         |
|156         |18.4133          |3.4374                   |15        |
|132         |15.9606          |3.4374                   |1856382   |
|86          |15.7763          |3.4374                   |3102      |
|23          |15.2245          |3.4374                   |40        |
|2           |14.9008          |3.4374                   |52        |
|117         |14.4493          |3.4374                   |2903      |
|115         |13.929           |3.4374                   |10        |
|201         |13.9086          |3.4374                   |793       |
|118         |13.8372          |3.4374                   |18        |
|46          |13.5657          |3.4374                   |187       |
|219         |13.381

### 3.2 CASE Expressions


In [49]:
time_of_day_summary_sql = spark.sql("""
    WITH classified AS (
        SELECT
            CASE
                WHEN hour_of_day BETWEEN 6 AND 11 THEN 'morning'
                WHEN hour_of_day BETWEEN 12 AND 17 THEN 'afternoon'
                WHEN hour_of_day BETWEEN 18 AND 22 THEN 'evening'
                ELSE 'night'
            END AS time_of_day,
            trip_distance,
            total_amount,
            tip_percentage
        FROM trips
    )
    SELECT
        time_of_day,
        ROUND(AVG(trip_distance), 4) AS avg_trip_distance,
        ROUND(AVG(total_amount), 4) AS avg_total_amount,
        ROUND(AVG(tip_percentage), 4) AS avg_tip_percentage,
        COUNT(*) AS trip_count
    FROM classified
    GROUP BY time_of_day
    ORDER BY avg_total_amount DESC
""")

time_of_day_summary_sql.show(truncate=False)


[Stage 187:================>                                      (5 + 12) / 17]

+-----------+-----------------+----------------+------------------+----------+
|time_of_day|avg_trip_distance|avg_total_amount|avg_tip_percentage|trip_count|
+-----------+-----------------+----------------+------------------+----------+
|afternoon  |3.4165           |30.0013         |20.3475           |13168208  |
|night      |3.9626           |29.4922         |21.4487           |4006397   |
|evening    |3.3407           |28.7178         |22.5501           |10782325  |
|morning    |3.335            |27.4768         |19.9199           |7666634   |
+-----------+-----------------+----------------+------------------+----------+



### 3.3 Window Functions: Ranking


In [50]:
monthly_top_5_pickups_sql = spark.sql("""
    WITH monthly_location_revenue AS (
        SELECT
            month,
            PULocationID,
            SUM(total_amount) AS total_revenue
        FROM trips
        GROUP BY month, PULocationID
    ),
    ranked AS (
        SELECT
            month,
            PULocationID,
            total_revenue,
            ROW_NUMBER() OVER (
                PARTITION BY month
                ORDER BY total_revenue DESC
            ) AS revenue_rank
        FROM monthly_location_revenue
    )
    SELECT
        month,
        PULocationID,
        ROUND(total_revenue, 4) AS total_revenue,
        revenue_rank
    FROM ranked
    WHERE revenue_rank <= 5
    ORDER BY month, revenue_rank
""")

monthly_top_5_pickups_sql.show(60, truncate=False)


[Stage 190:================>                                      (5 + 12) / 17]

+-----+------------+-------------+------------+
|month|PULocationID|total_revenue|revenue_rank|
+-----+------------+-------------+------------+
|1    |132         |1.107120876E7|1           |
|1    |138         |5745056.44   |2           |
|1    |161         |3234330.9    |3           |
|1    |237         |2672667.11   |4           |
|1    |230         |2657711.0    |5           |
|2    |132         |9743008.95   |1           |
|2    |138         |5644524.37   |2           |
|2    |161         |3345928.0    |3           |
|2    |237         |2666334.81   |4           |
|2    |230         |2562807.57   |5           |
|3    |132         |1.231711663E7|1           |
|3    |138         |7040654.95   |2           |
|3    |161         |3641500.23   |3           |
|3    |230         |3056386.32   |4           |
|3    |237         |2913362.74   |5           |
|4    |132         |1.222248976E7|1           |
|4    |138         |7297918.37   |2           |
|4    |161         |3682418.4    |3     

In [51]:
monthly_revenue_quartiles_sql = spark.sql("""
    WITH monthly_location_revenue AS (
        SELECT
            month,
            PULocationID,
            SUM(total_amount) AS total_revenue
        FROM trips
        GROUP BY month, PULocationID
    ),
    quartiles AS (
        SELECT
            month,
            PULocationID,
            total_revenue,
            NTILE(4) OVER (
                PARTITION BY month
                ORDER BY total_revenue DESC
            ) AS revenue_quartile
        FROM monthly_location_revenue
    )
    SELECT
        month,
        revenue_quartile,
        COUNT(*) AS location_count,
        ROUND(MIN(total_revenue), 4) AS min_revenue,
        ROUND(MAX(total_revenue), 4) AS max_revenue
    FROM quartiles
    GROUP BY month, revenue_quartile
    ORDER BY month, revenue_quartile
""")

monthly_revenue_quartiles_sql.show(48, truncate=False)

print("Top quartile location counts by month:")
monthly_revenue_quartiles_sql.filter("revenue_quartile = 1").show(12, truncate=False)


+-----+----------------+--------------+-----------+-------------+
|month|revenue_quartile|location_count|min_revenue|max_revenue  |
+-----+----------------+--------------+-----------+-------------+
|1    |1               |64            |40018.53   |1.107120876E7|
|1    |2               |63            |7525.68    |38020.78     |
|1    |3               |63            |2885.54    |7322.36      |
|1    |4               |63            |1.0        |2771.66      |
|2    |1               |63            |40866.86   |9743008.95   |
|2    |2               |63            |7869.75    |40029.74     |
|2    |3               |63            |2777.97    |7865.56      |
|2    |4               |63            |12.2       |2694.4       |
|3    |1               |63            |50864.59   |1.231711663E7|
|3    |2               |63            |10252.61   |48538.16     |
|3    |3               |63            |3522.64    |9963.11      |
|3    |4               |63            |8.4        |3521.74      |
|4    |1  

+-----+----------------+--------------+-----------+-------------+
|month|revenue_quartile|location_count|min_revenue|max_revenue  |
+-----+----------------+--------------+-----------+-------------+
|1    |1               |64            |40018.53   |1.107120876E7|
|2    |1               |63            |40866.86   |9743008.95   |
|3    |1               |63            |50864.59   |1.231711663E7|
|4    |1               |63            |53108.7    |1.222248976E7|
|5    |1               |63            |66280.83   |1.301141339E7|
|6    |1               |63            |61524.4    |1.254562764E7|
|7    |1               |64            |58712.62   |1.391994968E7|
|8    |1               |63            |62694.11   |1.389756513E7|
|9    |1               |63            |69411.83   |1.466673997E7|
|10   |1               |65            |61596.17   |1.520753598E7|
|11   |1               |64            |53895.01   |1.258497246E7|
|12   |1               |63            |64804.28   |1.247721028E7|
+-----+---

### 3.4 Window Functions: Running Totals and Moving Averages


In [53]:
daily_revenue_windows_sql = spark.sql("""
    WITH daily_revenue AS (
        SELECT
            TO_DATE(tpep_pickup_datetime) AS pickup_date,
            SUM(total_amount) AS daily_total_revenue
        FROM trips
        GROUP BY TO_DATE(tpep_pickup_datetime)
    )
    SELECT
        pickup_date,
        ROUND(daily_total_revenue, 4) AS daily_total_revenue,
        ROUND(
            SUM(daily_total_revenue) OVER (
                ORDER BY pickup_date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ),
            4
        ) AS running_total_revenue,
        ROUND(
            AVG(daily_total_revenue) OVER (
                ORDER BY pickup_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ),
            4
        ) AS seven_day_moving_avg_revenue,
        ROUND(
            daily_total_revenue - LAG(daily_total_revenue) OVER (ORDER BY pickup_date),
            4
        ) AS day_over_day_revenue_change
    FROM daily_revenue
    ORDER BY pickup_date
""")

daily_revenue_windows_sql.createOrReplaceTempView("daily_revenue_windows")
daily_revenue_windows_sql.select(
    "pickup_date",
    F.format_number("daily_total_revenue", 4).alias("daily_total_revenue"),
    F.format_number("running_total_revenue", 4).alias("running_total_revenue"),
    F.format_number("seven_day_moving_avg_revenue", 4).alias("seven_day_moving_avg_revenue"),
    F.format_number("day_over_day_revenue_change", 4).alias("day_over_day_revenue_change")
).show(20, truncate=False)


26/04/26 14:53:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:53:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:53:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:53:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:53:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:53:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 1

+-----------+-------------------+---------------------+----------------------------+---------------------------+
|pickup_date|daily_total_revenue|running_total_revenue|seven_day_moving_avg_revenue|day_over_day_revenue_change|
+-----------+-------------------+---------------------+----------------------------+---------------------------+
|2024-01-01 |2,102,786.4000     |2,102,786.4000       |2,102,786.4000              |NULL                       |
|2024-01-02 |2,179,962.0900     |4,282,748.4900       |2,141,374.2450              |77,175.6900                |
|2024-01-03 |2,265,092.2500     |6,547,840.7400       |2,182,613.5800              |85,130.1600                |
|2024-01-04 |2,687,180.7800     |9,235,021.5200       |2,308,755.3800              |422,088.5300               |
|2024-01-05 |2,600,641.5900     |11,835,663.1100      |2,367,132.6220              |-86,539.1900               |
|2024-01-06 |2,279,012.1900     |14,114,675.3000      |2,352,445.8833              |-321,629.400

In [54]:
biggest_single_day_drop_sql = spark.sql("""
    SELECT
        pickup_date,
        daily_total_revenue,
        day_over_day_revenue_change
    FROM daily_revenue_windows
    WHERE day_over_day_revenue_change IS NOT NULL
    ORDER BY day_over_day_revenue_change ASC
    LIMIT 1
""")

biggest_single_day_drop_sql.show(truncate=False)


26/04/26 14:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:55:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:55:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 14:55:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 1

+-----------+-------------------+---------------------------+
|pickup_date|daily_total_revenue|day_over_day_revenue_change|
+-----------+-------------------+---------------------------+
|2024-11-28 |1637755.11         |-1116131.88                |
+-----------+-------------------+---------------------------+



### 3.5 Window Functions: Comparisons


In [55]:
trips_above_hourly_average_sql = spark.sql("""
    SELECT
        tpep_pickup_datetime,
        PULocationID,
        DOLocationID,
        hour_of_day,
        ROUND(total_amount, 4) AS total_amount,
        ROUND(
            AVG(total_amount) OVER (PARTITION BY hour_of_day),
            4
        ) AS hourly_avg_total_amount,
        ROUND(
            total_amount - AVG(total_amount) OVER (PARTITION BY hour_of_day),
            4
        ) AS amount_above_hourly_avg
    FROM trips
    ORDER BY amount_above_hourly_avg DESC
    LIMIT 20
""")

trips_above_hourly_average_sql.show(20, truncate=False)


+--------------------+------------+------------+-----------+------------+-----------------------+-----------------------+
|tpep_pickup_datetime|PULocationID|DOLocationID|hour_of_day|total_amount|hourly_avg_total_amount|amount_above_hourly_avg|
+--------------------+------------+------------+-----------+------------+-----------------------+-----------------------+
|2024-12-04 14:24:18 |162         |186         |14         |3037.1      |30.1028                |3006.9972              |
|2024-07-22 20:41:53 |7           |7           |20         |2265.45     |28.4487                |2237.0013              |
|2024-01-14 10:08:11 |220         |220         |10         |2225.3      |27.3184                |2197.9816              |
|2024-04-06 16:33:28 |234         |226         |16         |1737.18     |32.0753                |1705.1047              |
|2024-11-06 09:05:55 |94          |48          |9          |1477.68     |26.7964                |1450.8836              |
|2024-06-30 18:20:04 |13

In [56]:
monthly_high_low_daily_revenue_sql = spark.sql("""
    WITH daily_revenue AS (
        SELECT
            month,
            TO_DATE(tpep_pickup_datetime) AS pickup_date,
            SUM(total_amount) AS daily_total_revenue
        FROM trips
        GROUP BY month, TO_DATE(tpep_pickup_datetime)
    ),
    annotated AS (
        SELECT
            month,
            FIRST_VALUE(pickup_date) OVER (
                PARTITION BY month
                ORDER BY daily_total_revenue DESC, pickup_date ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            ) AS highest_revenue_date,
            FIRST_VALUE(daily_total_revenue) OVER (
                PARTITION BY month
                ORDER BY daily_total_revenue DESC, pickup_date ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            ) AS highest_daily_revenue,
            LAST_VALUE(pickup_date) OVER (
                PARTITION BY month
                ORDER BY daily_total_revenue DESC, pickup_date ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            ) AS lowest_revenue_date,
            LAST_VALUE(daily_total_revenue) OVER (
                PARTITION BY month
                ORDER BY daily_total_revenue DESC, pickup_date ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            ) AS lowest_daily_revenue
        FROM daily_revenue
    )
    SELECT DISTINCT
        month,
        highest_revenue_date,
        ROUND(highest_daily_revenue, 4) AS highest_daily_revenue,
        lowest_revenue_date,
        ROUND(lowest_daily_revenue, 4) AS lowest_daily_revenue
    FROM annotated
    ORDER BY month
""")

monthly_high_low_daily_revenue_sql.show(12, truncate=False)


[Stage 229:===================================>                   (11 + 6) / 17]

+-----+--------------------+---------------------+-------------------+--------------------+
|month|highest_revenue_date|highest_daily_revenue|lowest_revenue_date|lowest_daily_revenue|
+-----+--------------------+---------------------+-------------------+--------------------+
|1    |2024-01-25          |2827199.56           |2024-01-07         |1799281.66          |
|2    |2024-02-29          |3175067.09           |2024-02-13         |1636503.43          |
|3    |2024-03-14          |3355273.76           |2024-03-31         |2303892.27          |
|4    |2024-04-18          |3358428.49           |2024-04-22         |2420538.54          |
|5    |2024-05-16          |3645497.0            |2024-05-27         |1895569.04          |
|6    |2024-06-06          |3584550.74           |2024-06-30         |2208535.4           |
|7    |2024-07-18          |3162952.41           |2024-07-04         |1489733.13          |
|8    |2024-08-01          |2959176.55           |2024-08-18         |2103687.65

### 3.6 ROLLUP


In [57]:
rollup_revenue_sql = spark.sql("""
    SELECT
        month,
        payment_type,
        ROUND(SUM(total_amount), 4) AS total_revenue,
        COUNT(*) AS trip_count,
        CASE
            WHEN month IS NULL AND payment_type IS NULL THEN 'grand total'
            WHEN payment_type IS NULL THEN 'monthly subtotal'
            ELSE 'month/payment_type detail'
        END AS rollup_row_type
    FROM trips
    GROUP BY ROLLUP(month, payment_type)
    ORDER BY month, payment_type
""")

rollup_revenue_sql.show(100, truncate=False)


[Stage 235:======================>                                (7 + 10) / 17]

+-----+------------+-----------------+----------+-------------------------+
|month|payment_type|total_revenue    |trip_count|rollup_row_type          |
+-----+------------+-----------------+----------+-------------------------+
|NULL |NULL        |1.0335206038687E9|35623564  |grand total              |
|1    |NULL        |7.46182109899E7  |2723184   |monthly subtotal         |
|1    |1           |6.385326936E7    |2273105   |month/payment_type detail|
|1    |2           |9979769.85       |417798    |month/payment_type detail|
|1    |3           |217962.16        |9697      |month/payment_type detail|
|1    |4           |567209.62        |22584     |month/payment_type detail|
|2    |NULL        |7.44438453399E7  |2719133   |monthly subtotal         |
|2    |1           |6.434241822E7    |2294733   |month/payment_type detail|
|2    |2           |9293911.06       |391648    |month/payment_type detail|
|2    |3           |221516.05        |9603      |month/payment_type detail|
|2    |4    

In the `ROLLUP` output, rows where `payment_type` is `NULL` are monthly subtotals across all payment types. The row where both `month` and `payment_type` are `NULL` is the grand total for the whole cleaned dataset.


## Part 4: Performance Analysis

This section inspects Spark query plans and measures the effect of caching and partition pruning on the cleaned taxi dataset. The benchmark query is the same daily revenue window query from Part 3.4, but it uses a narrow temporary view with only the columns required for that query so caching does not try to keep the entire wide taxi table in memory.


In [1]:
from time import perf_counter
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

try:
    spark
except NameError:
    spark = (
        SparkSession.builder
        .appName("Assignment2")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

clean_output_path = "data/nyc_taxi_2024_clean.parquet"

# Read a fresh copy from disk for performance tests. This avoids accidental reuse
# of cached state from earlier cells.
df_clean_perf = spark.read.parquet(clean_output_path)

# More shuffle partitions reduce per-task aggregation memory pressure.
spark.conf.set("spark.sql.shuffle.partitions", "64")

def time_action(label, action):
    start = perf_counter()
    result = action()
    elapsed = perf_counter() - start
    print(f"{label}: {elapsed:.2f} seconds")
    return elapsed, result

print(f"Performance analysis dataset rows: {df_clean_perf.count():,}")
print(f"Shuffle partitions for Part 4: {spark.conf.get('spark.sql.shuffle.partitions')}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 15:55:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 1:>                                                        (0 + 16) / 16]

Performance analysis dataset rows: 35,623,564
Shuffle partitions for Part 4: 64


### 4.1 Query Plans


In [2]:
trips_perf_base = df_clean_perf.select(
    "tpep_pickup_datetime",
    "total_amount",
)
trips_perf_base.createOrReplaceTempView("trips_perf")

daily_revenue_performance_query = """
    WITH daily_revenue AS (
        SELECT
            TO_DATE(tpep_pickup_datetime) AS pickup_date,
            SUM(total_amount) AS daily_total_revenue
        FROM trips_perf
        GROUP BY TO_DATE(tpep_pickup_datetime)
    )
    SELECT
        pickup_date,
        daily_total_revenue,
        SUM(daily_total_revenue) OVER (
            ORDER BY pickup_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total_revenue,
        AVG(daily_total_revenue) OVER (
            ORDER BY pickup_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS seven_day_moving_avg_revenue,
        daily_total_revenue - LAG(daily_total_revenue) OVER (ORDER BY pickup_date)
            AS day_over_day_revenue_change
    FROM daily_revenue
    ORDER BY pickup_date
"""

daily_revenue_plan_df = spark.sql(daily_revenue_performance_query)
daily_revenue_plan_df.explain(True)


== Parsed Logical Plan ==
CTE [daily_revenue]
:  +- 'SubqueryAlias daily_revenue
:     +- 'Aggregate ['TO_DATE('tpep_pickup_datetime)], ['TO_DATE('tpep_pickup_datetime) AS pickup_date#58, 'SUM('total_amount) AS daily_total_revenue#59]
:        +- 'UnresolvedRelation [trips_perf], [], false
+- 'Sort ['pickup_date ASC NULLS FIRST], true
   +- 'Project ['pickup_date, 'daily_total_revenue, 'SUM('daily_total_revenue) windowspecdefinition('pickup_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS running_total_revenue#55, 'AVG('daily_total_revenue) windowspecdefinition('pickup_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS seven_day_moving_avg_revenue#56, ('daily_total_revenue - 'LAG('daily_total_revenue) windowspecdefinition('pickup_date ASC NULLS FIRST, unspecifiedframe$())) AS day_over_day_revenue_change#57]
      +- 'UnresolvedRelation [daily_revenue], [], false

== Analyzed Logical Plan ==
pickup_date: date, daily_

26/04/26 15:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 15:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


For the report, look for `Exchange` operators to identify shuffles, `HashAggregate` or `SortAggregate` for aggregation stages, `Window` for the running total and moving average, and the Parquet `ReadSchema` to confirm Spark reads only `tpep_pickup_datetime` and `total_amount` for this query.


### 4.2 Caching


In [3]:
def run_daily_revenue_query():
    return spark.sql(daily_revenue_performance_query).count()

spark.catalog.clearCache()
trips_perf_base.unpersist(blocking=True)
trips_perf_base.createOrReplaceTempView("trips_perf")

uncached_times = []
for run_number in range(1, 4):
    elapsed, row_count = time_action(
        f"Uncached run {run_number}",
        run_daily_revenue_query,
    )
    uncached_times.append(elapsed)
    print(f"Rows returned: {row_count:,}")

uncached_avg = sum(uncached_times) / len(uncached_times)
print(f"Average uncached time: {uncached_avg:.2f} seconds")


Uncached run 1: 2.86 seconds
Rows returned: 366


Uncached run 2: 1.67 seconds
Rows returned: 366


[Stage 16:============================>                            (8 + 8) / 16]

Uncached run 3: 1.50 seconds
Rows returned: 366
Average uncached time: 2.01 seconds


In [4]:
spark.catalog.clearCache()

# Cache the base DataFrame for the aggregation query, but keep it narrow.
# Caching the entire cleaned taxi table can exceed the available memory.
trips_perf_cached = trips_perf_base.cache()
trips_perf_cached.count()  # materialize the cache before timing
trips_perf_cached.createOrReplaceTempView("trips_perf")

cached_times = []
for run_number in range(1, 4):
    elapsed, row_count = time_action(
        f"Cached run {run_number}",
        run_daily_revenue_query,
    )
    cached_times.append(elapsed)
    print(f"Rows returned: {row_count:,}")

cached_avg = sum(cached_times) / len(cached_times)
print(f"Average cached time: {cached_avg:.2f} seconds")
print(f"Difference: {uncached_avg - cached_avg:.2f} seconds")

trips_perf_cached.unpersist(blocking=False)
trips_perf_base.createOrReplaceTempView("trips_perf")


Cached run 1: 1.09 seconds
Rows returned: 366
Cached run 2: 0.74 seconds
Rows returned: 366
Cached run 3: 0.58 seconds
Rows returned: 366
Average cached time: 0.80 seconds
Difference: 1.21 seconds


Use the averages above in the report. If caching does not improve the query, explain the observation: Parquet column pruning may already make reads efficient, the OS may cache file pages, and this query can still be dominated by shuffle, sort, and window execution rather than raw file reading.


### 4.3 Partitioned Reads


In [5]:
single_clean_path = "data/nyc_taxi_2024_clean.parquet"
partitioned_clean_path = "data/nyc_taxi_2024_clean_partitioned_by_month.parquet"

partition_write_df = df_clean_perf.select(
    "tpep_pickup_datetime",
    "total_amount",
    "month",
)

(
    partition_write_df
    .write
    .mode("overwrite")
    .partitionBy("month")
    .parquet(partitioned_clean_path)
)

print(f"Wrote month-partitioned data to {partitioned_clean_path}")


Wrote month-partitioned data to data/nyc_taxi_2024_clean_partitioned_by_month.parquet


In [6]:
target_month = 6

def count_target_month_from_single_file():
    return (
        spark.read.parquet(single_clean_path)
        .select("month", "total_amount")
        .filter(F.col("month") == target_month)
        .count()
    )

def count_target_month_from_partitioned_data():
    return (
        spark.read.parquet(partitioned_clean_path)
        .filter(F.col("month") == target_month)
        .count()
    )

single_elapsed, single_count = time_action(
    f"Single-file read filtered to month {target_month}",
    count_target_month_from_single_file,
)
partitioned_elapsed, partitioned_count = time_action(
    f"Partitioned read filtered to month {target_month}",
    count_target_month_from_partitioned_data,
)

print(f"Single-file count: {single_count:,}")
print(f"Partitioned count: {partitioned_count:,}")
print(f"Read-time difference: {single_elapsed - partitioned_elapsed:.2f} seconds")


Single-file read filtered to month 6: 1.26 seconds
Partitioned read filtered to month 6: 2.16 seconds
Single-file count: 3,006,250
Partitioned count: 3,006,250
Read-time difference: -0.90 seconds


In [7]:
single_month_df = (
    spark.read.parquet(single_clean_path)
    .select("month", "total_amount")
    .filter(F.col("month") == target_month)
)

partitioned_month_df = (
    spark.read.parquet(partitioned_clean_path)
    .filter(F.col("month") == target_month)
)

print("Single-file filtered read plan:")
single_month_df.explain(True)

print("Partitioned filtered read plan:")
partitioned_month_df.explain(True)


Single-file filtered read plan:
== Parsed Logical Plan ==
'Filter '`=`('month, 6)
+- Project [month#462, total_amount#456]
   +- Relation [VendorID#440,tpep_pickup_datetime#441,tpep_dropoff_datetime#442,passenger_count#443L,trip_distance#444,RatecodeID#445L,store_and_fwd_flag#446,PULocationID#447,DOLocationID#448,payment_type#449L,fare_amount#450,extra#451,mta_tax#452,tip_amount#453,tolls_amount#454,improvement_surcharge#455,total_amount#456,congestion_surcharge#457,Airport_fee#458,trip_duration_minutes#459,hour_of_day#460,day_of_week#461,month#462,tip_percentage#463,speed_mph#464] parquet

== Analyzed Logical Plan ==
month: int, total_amount: double
Filter (month#462 = 6)
+- Project [month#462, total_amount#456]
   +- Relation [VendorID#440,tpep_pickup_datetime#441,tpep_dropoff_datetime#442,passenger_count#443L,trip_distance#444,RatecodeID#445L,store_and_fwd_flag#446,PULocationID#447,DOLocationID#448,payment_type#449L,fare_amount#450,extra#451,mta_tax#452,tip_amount#453,tolls_amount#4

For the report, compare the two timings and inspect the partitioned plan for `PartitionFilters`. That confirms Spark can skip non-matching month folders instead of scanning the full dataset.
